# 🚢 Titanic Classification — Step by Step for Hamid

Hamid, in this notebook you will use a passenger's information to predict whether they survived.

**How to run:** Run Code cells from top to bottom with **Shift + Enter**. Markdown cells contain explanations. Try each exercise before reading the solution in the next cell. To start again, restart the runtime and run all cells in order.

Our learning sequence: **data → X/y → train/test split → create model → fit → predict → evaluate → new passenger**

You do not need a local CSV. If `titanic.csv` is missing, the notebook loads the data from the internet. Enable Internet in Kaggle for this option. If you are offline, add the CSV and change its path in the loading cell.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hghaderi-54/IntroAI/blob/main/Titanic/beginner/Titanic_Classification_Beginner_EN.ipynb)

**Open and run:** The badge opens this notebook from GitHub in Colab. You can also upload the file directly to Colab or use Kaggle's Import/Upload Notebook option. GitHub displays the notebook; a notebook environment runs its code.

## 1. What is classification?

In **regression**, we predict a number, such as a house price. In **classification**, we predict a category.

The `Survived` column contains each passenger's real answer:
- `0`: did not survive.
- `1`: survived.

These numbers are category labels. We are not predicting a score or price. This is a learning exercise; a model's prediction about a person is not certain.

✏️ **Exercise:** Which task is classification: predicting an exam score or predicting pass/fail?

<details><summary>Solution</summary>Pass/fail is classification. Predicting an exam score is regression.</details>

## 2. Tools and data

`pandas` stores data in a table called a **DataFrame**. `scikit-learn` provides tools for building and evaluating models. These libraries are usually available in Colab and Kaggle.

If you see `ModuleNotFoundError`, add a cell containing `%pip install pandas scikit-learn matplotlib`, run it, and then start again from the top.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/SharifiZarchi/IntroAI/main/Session_05/Titanic/titanic.csv"
LOCAL_CSV = Path("titanic.csv")  # In Kaggle, you can use /kaggle/input/.../titanic.csv
source = LOCAL_CSV if LOCAL_CSV.exists() else DATA_URL
df = pd.read_csv(source)
print("Data source:", source)
display(df.head())

## 3. Take a first look at the data

Each **row** represents one passenger. Each **column** contains a feature or the answer. We will use just three **features (input information)**:
- `Pclass`: ticket class, with values 1, 2, or 3. A smaller number means a higher class.
- `Sex`: sex as recorded in the dataset.
- `Age`: the passenger's age.

`Survived` is the **target (the answer we want to predict)**. To keep this exercise simple, we will leave out columns such as names and ticket numbers.

`shape` gives the number of rows and columns. `isna().sum()` counts missing values in each column.

In [ ]:
print("Rows, columns:", df.shape)
display(df[["Pclass", "Sex", "Age", "Survived"]].head())
print("Missing values:")
print(df[["Pclass", "Sex", "Age", "Survived"]].isna().sum())
print("Passenger counts by label:")
print(df["Survived"].value_counts().sort_index())

In [ ]:
df["Survived"].value_counts().sort_index().plot.bar(color=["gray", "teal"], rot=0)
plt.xticks([0, 1], ["0: Did not survive", "1: Survived"])
plt.ylabel("Passenger count")
plt.title("Titanic survival labels")
plt.show()

✏️ **Exercise:** Use `head(3)` to display only the first three passengers. Which of our selected columns has missing values in the full dataset? Try before running the next cell.

In [ ]:
display(df.head(3))
print(df[["Pclass", "Sex", "Age", "Survived"]].isna().sum())

## 4. Simple preparation and X/y

The model needs numerical inputs. **Encoding** means converting categories into numbers. Here, we use 0 for `female` and 1 for `male`, stored in a new column called `Sex_encoded`. This is only a coding convention; it does not imply a ranking.

For simplicity, we drop rows with a missing answer, ticket class, or encoded sex. We will fill missing ages after the train/test split.

In [ ]:
data = df[["Pclass", "Sex", "Age", "Survived"]].copy()
data["Sex_encoded"] = data["Sex"].map({"female": 0, "male": 1})
data = data.dropna(subset=["Pclass", "Sex_encoded", "Survived"])
display(data.head())

**What information do we have? → X**: ticket class, encoded sex, and age.

**What do we want to know? → y**: whether the passenger survived.

`X` is a **2-D (two-dimensional) table**: one axis for passengers and one for features. Even if we choose only age, the model expects a table with “number of passengers × one feature.”

In `df[["Age"]]`, the inner brackets create a **list of column names**. The outer brackets select those columns from the table. The result is a DataFrame. By comparison, `df["Age"]` gives a **Series (a one-dimensional column)**.

For this problem, `y` has only one answer per passenger, so it is **1-D (one-dimensional)**. Do not include `Survived` in X: it is the answer we are trying to predict!

✏️ **Exercise:** Build X using `Pclass`, `Sex_encoded`, and `Age`. Build y using `Survived`. The next cell contains the solution.

In [ ]:
feature_columns = ["Pclass", "Sex_encoded", "Age"]
X = data[feature_columns].copy()
y = data["Survived"].astype(int)
print("X shape:", X.shape)
print("y shape:", y.shape)
print("One feature as a table:", data[["Age"]].shape)
print("One column as a Series:", data["Age"].shape)

With n passengers, X has shape `(n, 3)` and y has shape `(n,)`. The comma in `(n,)` means this shape has just one dimension; y does not have a hidden second column.

## 5. Train/test split — learning and testing

We use 80% of the data for **training** and hold back 20% for **testing**. The model must not see the test answers while learning.

`random_state=42` makes the random split repeatable; 42 is not a magic number. `stratify=y` keeps the proportions of labels 0 and 1 approximately the same in both groups.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
X_train = X_train.copy()
X_test = X_test.copy()
print("Training:", X_train.shape, y_train.shape)
print("Testing:", X_test.shape, y_test.shape)

### Fill missing ages

The **median** is the middle age after sorting the ages. We calculate it using only the training data. Then we reuse that same number for training, testing, and any new passenger.

This avoids using test information when preparing the model. Accidentally allowing test information into training is called **data leakage**.

In [ ]:
age_median = X_train["Age"].median()
X_train["Age"] = X_train["Age"].fillna(age_median)
X_test["Age"] = X_test["Age"].fillna(age_median)
print("Training age median:", age_median)
print("Missing training values:", X_train.isna().sum().sum())
print("Missing test values:", X_test.isna().sum().sum())

## 6. Create a model

We will use **LogisticRegression**. Despite “Regression” in its name, it is used for classification. It learns how the inputs relate to labels 0 and 1.

This line only creates the model; it has not learned anything yet. `max_iter=1000` allows enough steps for the learning process.

In [ ]:
model = LogisticRegression(max_iter=1000)

## 7. Fit — let the model learn

`fit(X_train, y_train)` means “look at these features and their correct answers, and learn from them.” We use only the training data here.

✏️ **Exercise:** What belongs in `model.fit(____, ____)`?

In [ ]:
model.fit(X_train, y_train)
print("Model trained!")

## 8. Predict

Now `predict(X_test)` guesses the answers for passengers who were not used for training. We do not pass y_test to the model. We keep those real answers for checking the predictions.

In [ ]:
y_pred = model.predict(X_test)
comparison = pd.DataFrame({"Actual": y_test, "Predicted": y_pred})
display(comparison.head(10))

## 9. Evaluate — check the results

**Accuracy** is the number of correct predictions divided by the total number of predictions. For example, 8 correct answers out of 10 means 80% accuracy.

We also make a very simple comparison: always predict the most common answer in the **training** data. This is our **baseline**. Comparing with it helps us judge whether the model is useful. Accuracy alone does not explain every kind of mistake.

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
majority_label = int(y_train.mode().iloc[0])
baseline_predictions = [majority_label] * len(y_test)
baseline_accuracy = accuracy_score(y_test, baseline_predictions)
print(f"Model accuracy: {accuracy:.1%}")
print(f"Always predict {majority_label}: {baseline_accuracy:.1%}")

A **confusion matrix** shows which predictions were right and which were wrong.

In the plot below, **rows are actual answers** and **columns are predictions**. Both use the order 0, then 1:
- Top left: actual 0, predicted 0 — correct.
- Top right: actual 0, predicted 1 — wrong.
- Bottom left: actual 1, predicted 0 — wrong.
- Bottom right: actual 1, predicted 1 — correct.

The main diagonal contains correct predictions. The other two positions contain mistakes.

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
ConfusionMatrixDisplay(cm, display_labels=[0, 1]).plot(cmap="Blues", colorbar=False)
plt.title("Rows: actual | Columns: predicted")
plt.show()
print("Correct predictions:", cm[0, 0] + cm[1, 1])
print("Wrong predictions:", cm[0, 1] + cm[1, 0])

✏️ **Exercise:** Calculate accuracy again: add the two diagonal counts and divide by the sum of all counts in the matrix.

In [ ]:
accuracy_from_matrix = (cm[0, 0] + cm[1, 1]) / cm.sum()
print(f"Accuracy from matrix: {accuracy_from_matrix:.1%}")

This result applies to this particular data split; it does not guarantee performance for every passenger. If we keep changing the model based on test results, the test is no longer an independent check.

## 10. Predict a new passenger

Imagine a 28-year-old female passenger with a second-class ticket. Use the same encoding as before: female is 0. The column names and order must match X_train exactly.

Even one passenger needs a **2-D table with shape `(1, 3)`**: one row and three features. We do not have an actual answer for this made-up passenger; the model produces a guess.

In [ ]:
new_passenger = pd.DataFrame(
    [{"Pclass": 2, "Sex_encoded": 0, "Age": 28}],
    columns=feature_columns
)
# Use the training median if the new passenger's age is missing.
new_passenger["Age"] = new_passenger["Age"].fillna(age_median)
display(new_passenger)
print("Shape:", new_passenger.shape)
new_prediction = int(model.predict(new_passenger)[0])
labels = {0: "Did not survive", 1: "Survived"}
print("Predicted label:", new_prediction)
print("Model prediction:", labels[new_prediction])

✏️ **Final exercise:** Create a 40-year-old male passenger with a third-class ticket and make a prediction. Does the prediction change? Changing one feature does not always change the predicted label. Add a Code cell and try it before reading the next solution.

In [ ]:
practice_passenger = pd.DataFrame(
    [{"Pclass": 3, "Sex_encoded": 1, "Age": 40}],
    columns=feature_columns
)
practice_passenger["Age"] = practice_passenger["Age"].fillna(age_median)
practice_prediction = int(model.predict(practice_passenger)[0])
print("Predicted label:", practice_prediction)
print("Model prediction:", labels[practice_prediction])

## Review for Hamid

1. `pd.read_csv` loaded the data into a table.
2. `X` held the 2-D features and `y` held the 1-D answers.
3. `train_test_split` separated learning from testing.
4. We filled missing ages using the training median.
5. `LogisticRegression` created the model and `fit` trained it.
6. `predict` guessed the labels; accuracy and the confusion matrix checked the results.
7. We predicted a new passenger using the same columns and preparation.

**For GitHub:** Commit this `.ipynb` file. You do not need a CSV when loading from the internet. The Colab badge points to this notebook in `hghaderi-54/IntroAI`. If you move the file or copy it to another repository, update the badge URL.

Data source: [IntroAI — Titanic CSV](https://raw.githubusercontent.com/SharifiZarchi/IntroAI/main/Session_05/Titanic/titanic.csv). This is a beginner exercise, not a promise of a competitive Kaggle ranking.